# Using an API to Communicate with a Model

This notebook follows the rendered lesson episodes in order. Run the cells from top to bottom.

## Introduction: What Is an LLM API?

### Questions

- What exactly is a "large language model API", and how is it different from running a model yourself?
- What does a request to this API look like under the hood?


### Objectives

- Explain the client / service relationship behind an LLM API.
- Describe the three things you choose in a request: the model, the messages, and the knobs.
- Locate the answer in the response JSON.


## You are a client talking to a service

The core idea of this lesson is one thing: **you send a message, you get a
message back** — over the internet, to a model running on someone else's
servers. You are a *client* talking to a *service*: the model is hosted for
you, so you simply send it text and read the structured text it returns.

You'll use a real one: **ASU Research Computing's LLM gateway**, the same
OpenAI-compatible API its own researchers use:

- **Endpoint:** [`https://openai.rc.asu.edu/v1`](https://openai.rc.asu.edu/v1)
- **SDK:** the standard [`openai` Python package](https://pypi.org/project/openai/)
- **Your key:** created in [Voyager](https://voyager.rc.asu.edu) and given to you for this workshop

The word **"OpenAI-compatible"** does a lot of work. It's a standard: any tool
that talks to OpenAI talks to this too. That single fact is what unlocks the
"tools around the API" at the end of the lesson.

## Anatomy of a request

Before we write code, see what a request actually is. The Python SDK builds
and sends a **JSON** document over the internet for you:

```python
from openai import OpenAI

client = OpenAI(
    api_key="***",
    base_url="https://openai.rc.asu.edu/v1",
)

resp = client.chat.completions.create(
    model="<MODEL_NAME>",
    messages=[
        {"role": "user", "content": "Explain quantum computing in one sentence."}
    ],
)
```

Three things you're always choosing:

1. **the `model`** — which model answers (from your key's live list),
2. **the `messages`** — a list of `{role, content}` pairs,
3. **optional settings** — like `temperature` (0 = as deterministic as the model allows).

The response is JSON too, and the text you actually want lives at
**`choices[0].message.content`**. The `openai` Python package builds this
request for you and turns the JSON response back into a Python object you can
read.

### The `role` field — the most useful concept today

- `system` = standing instructions (the kind of answer you want)
- `user` = your turn (the text you send)
- `assistant` = the model's reply

You'll set a `system` message later to force *structured* output.


### Key Points

- An LLM API is a *service*; you are a *client*. You send JSON, you get JSON back.
- You always choose the **model**, the **messages** (`role` + `content`), and the **knobs**.
- The answer is at `choices[0].message.content`.
- It's **OpenAI-compatible**, so the `openai` Python SDK (and many other tools) just work.

## Your First Call

### Questions

- How do I set my key without pasting it into my code?
- How do I make the very first call and know it worked?


### Objectives

- Put your key and endpoint in a `.env` file (never hardcode them).
- Make a first call with the `openai` Python package.
- Read the model's reply out of the response.


## Set your key (don't hardcode it)

Your key is a secret. Treat it like a password: **don't share it, don't commit
it to Git.** The notebook reads it from a **`.env` file** so it never sits in
your code. A `.env` file is plain text, one `NAME=value` line at a time.

In the **same folder as the notebook**, create a file named `.env` (no
extension) with exactly these two lines:

```
OPENAI_API_KEY=<paste your key here>
OPENAI_BASE_URL=https://openai.rc.asu.edu/v1
```

The [Setup page](../learners/setup.md) shows how to create that file on Anvil
and on a laptop. If you are working inside a notebook environment like Google
Colab, you can create the `.env` file from a notebook cell instead:

In [ ]:
from getpass import getpass
from pathlib import Path

key = getpass("Paste your API key, then press Enter: ").strip()
Path(".env").write_text(
    f"OPENAI_API_KEY={key}\n"
    "OPENAI_BASE_URL=https://openai.rc.asu.edu/v1\n"
)
del key
print("Wrote .env for this notebook session.")

Then run the **"Set your key"** cell in the notebook — it loads the file and
verifies the key is present (masked):

In [ ]:
import os
from dotenv import load_dotenv

# Load environment variables from the .env file
load_dotenv()

# Retrieve the API key and base URL
key = os.getenv("OPENAI_API_KEY")
base = os.getenv("OPENAI_BASE_URL")

if not key or not base:
    missing = ", ".join(n for n, v in
                        (("OPENAI_API_KEY", key), ("OPENAI_BASE_URL", base)) if not v)
    raise SystemExit(
        f"Missing {missing}. Create a .env file in this folder with these two "
        "lines, then re-run this cell:\n"
        "    OPENAI_API_KEY=<your-key>\n"
        "    OPENAI_BASE_URL=https://openai.rc.asu.edu/v1"
    )

# Mask the key so we can show it is present without revealing it
print(f"key set:   {key[:4]}...{key[-4:]}  ({len(key)} chars)")
print(f"base url:  {base}")

Two things to notice. `load_dotenv()` reads the `.env` file into the
environment, and `os.getenv()` looks a name up in that environment — so the
key only lives in the file, which you keep out of Git. Because the cell reads
the file every time it runs, **if you edit `.env` you only re-run the cell** —
no kernel restart.

## The first call — *checkpoint: everyone gets a response*


The next cell picks one of the three models this lesson prefers — **at random** —
from your key's live list, so you can run the first call right away. It also
prints up to eight models your key can use. If none of the three preferred
models is available, it falls back to the first model on the list; if nothing
prints at all, set `MODEL` yourself and re-run.

In [ ]:
from openai import OpenAI
import random

client = OpenAI()   # reads OPENAI_API_KEY and OPENAI_BASE_URL from the environment

# Pick a model at random from the live list for your key.
PREFERRED = ["qwen36-27b", "muse-glimmer-30b", "gemma4-31b-it"]
try:
    available = [m.id for m in client.models.list()]
except Exception as e:
    available = []
    print(f"Could not list models: {e}")

# A preferred name can be a prefix of the full model ID, so match by
# substring. Then pick one of the matches at random.
matches = [a for a in available if any(p in a for p in PREFERRED)]
if matches:
    MODEL = random.choice(matches)
elif available:
    MODEL = available[0]   # none of the three is available; use the first
else:
    raise SystemExit(
        "No model could be selected. Set one manually, e.g.\n"
        "    MODEL = 'llama3.1'\n"
        "then re-run."
    )
print(f"Using model: {MODEL}  (picked at random)")
if available:
    print(f"Your key can use {len(available)} model(s). Here are the first eight:")
    for a in available[:8]:
        print(f"   - {a}")

Now send the first request with the selected model:

In [ ]:
resp = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "Explain what an API is, in one sentence."}],
)
print("\n--- MODEL SAYS ---")
print(resp.choices[0].message.content)
print(f"\n(tokens used: {resp.usage.total_tokens if resp.usage else '?'})")

You should see a sentence from the model. If text appears, your first call worked.

### Try a setting: `temperature`

The API has small controls, often called *settings* or *knobs*, that change how
the model answers. `temperature` is a good first one to try: `0` asks for the
most consistent wording the model can give, while larger values usually make the
wording more varied.

Run the cell below a few times. Then change `temperature` to `0`, `0.7`, or
`1.0` and compare what happens.

In [ ]:
for temp in [0, 0.7, 1.0]:
    resp = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": "Give me a friendly one-sentence definition of an API."}],
        temperature=temp,
    )
    print(f"temperature={temp}")
    print(resp.choices[0].message.content)
    print()

## Anatomy of a response

Every reply comes back as a JSON object with the same shape. Here is a typical
one, with each field explained:

```json
{
  "id": "chatcmpl-2d4be949-...",            // 1 — unique ID for this
                                            //    response; send it to support
                                            //    when reporting a problem
  "object": "chat.completion",              // 2 — the kind of object this
                                            //    is (almost always this value
                                            //    for chat completions)
  "created": 1757984284,                    // 3 — Unix timestamp (seconds since
                                            //    1970) of when the response
                                            //    was created
  "model": "gemma4-31b-it",                 // 4 — the model that actually
                                            //    handled the request (can
                                            //    differ from what you asked
                                            //    for, if the gateway routed
                                            //    it elsewhere)
  "choices": [                              // 5 — the answer(s). A list
    {                                       //    because n > 1 can request
      "index": 0,                           //    several alternatives; 0
                                            //    for the first one
      "message": {                          //    the model's reply lives
        "role": "assistant",                //    inside "message"
        "content": "This study proposes..."//    the text you wanted
      },
      "finish_reason": "stop",              // 6 — why generation stopped:
                                            //    "stop" = model finished
                                            //    naturally, "length" = hit
                                            //    max_tokens, "content_filter"
                                            //    = safety filter tripped
      "logprobs": null                      //    per-token probabilities;
                                            //    null unless requested
    }
  ],
  "usage": {                                // 7 — what this call consumed
    "prompt_tokens": 12,                    //    tokens in your messages
    "completion_tokens": 87,                //    tokens in the reply
    "total_tokens": 99                      //    sum; billing is based on
                                            //    this
  }
}
```

Three fields matter most for the rest of the lesson:

1. **`choices[0].message.content`** — the actual text. This is the field every
   call in the rest of the lesson reads.
2. **`choices[0].finish_reason`** — if it says `"length"`, your reply was cut
   off at `max_tokens`; raise the limit and try again.
3. **`usage.total_tokens`** — the cost meter. The first cell above prints it
   (`resp.usage.total_tokens`); when you write your own batch tool, summing it
   per call tells you what a whole run cost.

### How does `resp` relate to this JSON?

The Python client unpacks the JSON into attributes: `resp.choices[0].message.content`
is the `choices[0].message.content` above, and `resp.usage` gives you the token
counts. Knowing the raw shape matters when you read documentation (which
describes the JSON, not the Python) or when you move to a language without a
client library.

### Two ideas to hold onto

- **`messages`** is a list of `{role, content}` (recap the `role` field from
  the introduction).
- The answer lives at **`resp.choices[0].message.content`** — field 5 in the
  anatomy above.

You can list the models your key can use — you'll use this again in the
experiments:

In [ ]:
print([m.id for m in client.models.list()])

### Stuck? Read the error verbatim

- `401` / "No api key" → the `.env` file is missing, in the wrong folder, or
  has a typo (a stray space or a `<` left in). Check both lines, then re-run
  the "Set your key" cell — the cell re-reads the file, so no restart is
  needed.
- `ModuleNotFoundError: No module named 'dotenv'` → you're outside the venv,
  or the install didn't include `python-dotenv`. Re-run
  `pip install openai python-dotenv` and restart the kernel.
- `404` / "Model Not Found" → bad model name. List models and copy an exact ID.
- `429` → rate limit. Wait a few seconds and retry.
- Connection/timeout/SSL → possible **egress** problem. See the [Reference
  page](../learners/reference.md) or raise a hand.

### Key Points

- Put the key and endpoint in a `.env` file; `load_dotenv()` reads it, so
  nothing is hardcoded.
- `client = OpenAI()` reads `OPENAI_API_KEY` and `OPENAI_BASE_URL` from the
  environment.
- The first call is `client.chat.completions.create(model=..., messages=[...])`.
- The answer is at `resp.choices[0].message.content`; `finish_reason` tells you
  why generation stopped and `usage.total_tokens` is the cost meter.

## Building the Research Tool

### Questions

- How do I run the *same* structured question over a *list* of texts and get a table back?
- What is "structured output", and why does it make an LLM useful for research?


### Objectives

- Batch-process a list of abstracts with one model call each.
- Use a `system` prompt to force strict JSON output.
- Write the structured rows to a CSV file.


The goal here is a research task: run the *same* question over a *list* of
things and get a *table* back. Here that means 12 paper abstracts in and one
row each out — a one-line summary, the main method, and the key result, in a
spreadsheet. Two techniques make it work: **batch API calls** (one model call
per abstract) and **structured output**.

Structured output comes from the **system prompt**: it tells the model to answer
in *strict JSON* with three keys — `summary`, `method`, `result`. Because you
specify the shape of the answer, each result comes back as clean columns you can
load straight into a spreadsheet.

Run the three cells in this order: load (3a) → batch (3b) → save (3c).

### 3a — Load the 12 abstracts (already in the notebook)

The 12 real arXiv abstracts (2 per field, 6 fields) are **already in the
notebook** and load directly from the cell. The first entry looks like this;
all 12 live in the notebook and in
[`data/research_abstracts.json`](data/research_abstracts.json):

In [ ]:
import json

ABSTRACTS = [
  {
    "field": "Machine Learning (CS)",
    "title": "Learning Active Subspaces and Discovering Important Features with Gaussian Radial Basis Functions Neural Networks",
    "abstract": "Providing a model that achieves a strong predictive performance and is simultaneously interpretable by humans is one of the most difficult challenges in machine learning research due to the conflicting nature of these two objectives. To address this challenge, we propose a modification of the radial basis function neural network model by equipping its Gaussian kernel with a learnable precision matrix. We show that precious information is contained in the spectrum of the precision matrix that can be extracted once the training of the model is completed. In particular, the eigenvectors explain the directions of maximum sensitivity of the model revealing the active subspace and suggesting potential applications for supervised dimensionality reduction. At the same time, the eigenvectors highlight the relationship in terms of absolute variation between the input and the latent variables, thereby allowing us to extract a ranking of the input variables based on their importance to the prediction task enhancing the model interpretability. We conducted numerical experiments for regression, classification, and feature selection tasks, comparing our model against popular machine learning models, the state-of-the-art deep learning-based embedding feature selection techniques, and a transformer model for tabular data. Our results demonstrate that the proposed model does not only yield an attractive prediction performance compared to the competitors but also provides meaningful and interpretable results that potentially could assist the decision-making process in real-world applications. A PyTorch implementation of the model is available on GitHub at the following link. https://github.com/dannyzx/Gaussian-RBFNN",
    "id": "2307.05639v2"
  },
  {
    "field": "Machine Learning (CS)",
    "title": "Hierarchical Attentional Hybrid Neural Networks for Document Classification",
    "abstract": "Document classification is a challenging task with important applications. The deep learning approaches to the problem have gained much attention recently. Despite the progress, the proposed models do not incorporate the knowledge of the document structure in the architecture efficiently and not take into account the contexting importance of words and sentences. In this paper, we propose a new approach based on a combination of convolutional neural networks, gated recurrent units, and attention mechanisms for document classification tasks. The main contribution of this work is the use of convolution layers to extract more meaningful, generalizable and abstract features by the hierarchical representation. The proposed method in this paper improves the results of the current attention-based approaches for document classification.",
    "id": "1901.06610v2"
  },
  {
    "field": "Biology",
    "title": "Learning differential module networks across multiple experimental conditions",
    "abstract": "Module network inference is a statistical method to reconstruct gene regulatory networks, which uses probabilistic graphical models to learn modules of coregulated genes and their upstream regulatory programs from genome-wide gene expression and other omics data. Here we review the basic theory of module network inference, present protocols for common gene regulatory network reconstruction scenarios based on the Lemon-Tree software, and show, using human gene expression data, how the software can also be applied to learn differential module networks across multiple experimental conditions.",
    "id": "1711.08927v2"
  },
  {
    "field": "Biology",
    "title": "Gene regulatory network inference: an introductory survey",
    "abstract": "Gene regulatory networks are powerful abstractions of biological systems. Since the advent of high-throughput measurement technologies in biology in the late 90s, reconstructing the structure of such networks has been a central computational problem in systems biology. While the problem is certainly not solved in its entirety, considerable progress has been made in the last two decades, with mature tools now available. This chapter aims to provide an introduction to the basic concepts underpinning network inference tools, attempting a categorisation which highlights commonalities and relative strengths. While the chapter is meant to be self-contained, the material presented should provide a useful background to the later, more specialised chapters of this book.",
    "id": "1801.04087v2"
  },
  {
    "field": "Materials Science",
    "title": "The Efficiency Limit of CH3NH3PbI3 Perovskite Solar Cells",
    "abstract": "With the consideration of photon recycling effect, the efficiency limit of methylammonium lead iodide (CH3NH3PbI3) perovskite solar cells is predicted by a detailed balance model. To obtain convincing predictions, both AM 1.5 spectrum of Sun and experimentally measured complex refractive index of perovskite material are employed in the detailed balance model. The roles of light trapping and angular restriction in improving the maximal output power of thin-film perovskite solar cells are also clarified. The efficiency limit of perovskite cells (without the angular restriction) is about 31%, which approaches to Shockley-Queisser limit (33%) achievable by gallium arsenide (GaAs) cells. Moreover, the Shockley-Queisser limit could be reached with a 200 nm-thick perovskite solar cell, through integrating a wavelength-dependent angular-restriction design with a textured light-trapping structure. Additionally, the influence of the trap-assisted nonradiative recombination on the device efficiency is investigated. The work is fundamentally important to high-performance perovskite photovoltaics.",
    "id": "1506.09003v1"
  },
  {
    "field": "Materials Science",
    "title": "Towards the maximum efficiency design of a perovskite solar cell by material properties tuning: A multidimensional approach",
    "abstract": "To obtain significant increases in the Power Conversion Efficiency (PCE) of solar cells, future cell research and development should be based on the concomitant improvement of multiple material properties, rather than on the state-of-the-art one or two-dimensional improvements. In this context, researchers should know, which combined material properties and cell design parameters lead to the highest efficiency increase. For the same objective, it should also be known which relationships in-between these variables have to be adjusted. Such knowledge becomes available by simulation and numerical optimization, which we present for a Perovskite Solar Cell(PSC)in a hypercube space of variables.",
    "id": "1711.03818v2"
  },
  {
    "field": "Environmental Science",
    "title": "Predicting concentration levels of air pollutants by transfer learning and recurrent neural network",
    "abstract": "Air pollution (AP) poses a great threat to human health, and people are paying more attention than ever to its prediction. Accurate prediction of AP helps people to plan for their outdoor activities and aids protecting human health. In this paper, long-short term memory (LSTM) recurrent neural networks (RNNs) have been used to predict the future concentration of air pollutants (APS) in Macau. Additionally, meteorological data and data on the concentration of APS have been utilized. Moreover, in Macau, some air quality monitoring stations (AQMSs) have less observed data in quantity, and, at the same time, some AQMSs recorded less observed data of certain types of APS. Therefore, the transfer learning and pre-trained neural networks have been employed to assist AQMSs with less observed data to build a neural network with high prediction accuracy. The experimental sample covers a period longer than 12-year and includes daily measurements from several APS as well as other more classical meteorological values. Records from five stations, four out of them are AQMSs and the remaining one is an automatic weather station, have been prepared from the aforesaid period and eventually underwent to computational intelligence techniques to build and extract a prediction knowledge-based system. As shown by experimentation, LSTM RNNs initialized with transfer learning methods have higher prediction accuracy; it incurred shorter training time than randomly initialized recurrent neural networks.",
    "id": "2502.01654v1"
  },
  {
    "field": "Environmental Science",
    "title": "Urban Air Pollution Forecasting: a Machine Learning Approach leveraging Satellite Observations and Meteorological Forecasts",
    "abstract": "Air pollution poses a significant threat to public health and well-being, particularly in urban areas. This study introduces a series of machine-learning models that integrate data from the Sentinel-5P satellite, meteorological conditions, and topological characteristics to forecast future levels of five major pollutants. The investigation delineates the process of data collection, detailing the combination of diverse data sources utilized in the study. Through experiments conducted in the Milan metropolitan area, the models demonstrate their efficacy in predicting pollutant levels for the forthcoming day, achieving a percentage error of around 30%. The proposed models are advantageous as they are independent of monitoring stations, facilitating their use in areas without existing infrastructure. Additionally, we have released the collected dataset to the public, aiming to stimulate further research in this field. This research contributes to advancing our understanding of urban air quality dynamics and emphasizes the importance of amalgamating satellite, meteorological, and topographical data to develop robust pollution forecasting models.",
    "id": "2405.19901v1"
  },
  {
    "field": "Medicine / Health",
    "title": "Segmentation-Renormalized Deep Feature Modulation for Unpaired Image Harmonization",
    "abstract": "Deep networks are now ubiquitous in large-scale multi-center imaging studies. However, the direct aggregation of images across sites is contraindicated for downstream statistical and deep learning-based image analysis due to inconsistent contrast, resolution, and noise. To this end, in the absence of paired data, variations of Cycle-consistent Generative Adversarial Networks have been used to harmonize image sets between a source and target domain. Importantly, these methods are prone to instability, contrast inversion, intractable manipulation of pathology, and steganographic mappings which limit their reliable adoption in real-world medical imaging. In this work, based on an underlying assumption that morphological shape is consistent across imaging sites, we propose a segmentation-renormalized image translation framework to reduce inter-scanner heterogeneity while preserving anatomical layout. We replace the affine transformations used in the normalization layers within generative networks with trainable scale and shift parameters conditioned on jointly learned anatomical segmentation embeddings to modulate features at every level of translation. We evaluate our methodologies against recent baselines across several imaging modalities (T1w MRI, FLAIR MRI, and OCT) on datasets with and without lesions. Segmentation-renormalization for translation GANs yields superior image harmonization as quantified by Inception distances, demonstrates improved downstream utility via post-hoc segmentation accuracy, and improved robustness to translation perturbation and self-adversarial attacks.",
    "id": "2102.06315v2"
  },
  {
    "field": "Medicine / Health",
    "title": "PSIGAN: Joint probabilistic segmentation and image distribution matching for unpaired cross-modality adaptation based MRI segmentation",
    "abstract": "We developed a new joint probabilistic segmentation and image distribution matching generative adversarial network (PSIGAN) for unsupervised domain adaptation (UDA) and multi-organ segmentation from magnetic resonance (MRI) images. Our UDA approach models the co-dependency between images and their segmentation as a joint probability distribution using a new structure discriminator. The structure discriminator computes structure of interest focused adversarial loss by combining the generated pseudo MRI with probabilistic segmentations produced by a simultaneously trained segmentation sub-network. The segmentation sub-network is trained using the pseudo MRI produced by the generator sub-network. This leads to a cyclical optimization of both the generator and segmentation sub-networks that are jointly trained as part of an end-to-end network. Extensive experiments and comparisons against multiple state-of-the-art methods were done on four different MRI sequences totalling 257 scans for generating multi-organ and tumor segmentation. The experiments included, (a) 20 T1-weighted (T1w) in-phase mdixon and (b) 20 T2-weighted (T2w) abdominal MRI for segmenting liver, spleen, left and right kidneys, (c) 162 T2-weighted fat suppressed head and neck MRI (T2wFS) for parotid gland segmentation, and (d) 75 T2w MRI for lung tumor segmentation. Our method achieved an overall average DSC of 0.87 on T1w and 0.90 on T2w for the abdominal organs, 0.82 on T2wFS for the parotid glands, and 0.77 on T2w MRI for lung tumors.",
    "id": "2007.09465v2"
  },
  {
    "field": "Energy",
    "title": "Lasso estimation for GEFCom2014 probabilistic electric load forecasting",
    "abstract": "We present a methodology for probabilistic load forecasting that is based on lasso (least absolute shrinkage and selection operator) estimation. The model considered can be regarded as a bivariate time-varying threshold autoregressive(AR) process for the hourly electric load and temperature. The joint modeling approach incorporates the temperature effects directly, and reflects daily, weekly, and annual seasonal patterns and public holiday effects. We provide two empirical studies, one based on the probabilistic load forecasting track of the Global Energy Forecasting Competition 2014 (GEFCom2014-L), and the other based on another recent probabilistic load forecasting competition that follows a setup similar to that of GEFCom2014-L. In both empirical case studies, the proposed methodology outperforms two multiple linear regression based benchmarks from among the top eight entries to GEFCom2014-L.",
    "id": "1603.01376v1"
  },
  {
    "field": "Energy",
    "title": "A comparative assessment of deep learning models for day-ahead load forecasting: Investigating key accuracy drivers",
    "abstract": "Short-term load forecasting (STLF) is vital for the effective and economic operation of power grids and energy markets. However, the non-linearity and non-stationarity of electricity demand as well as its dependency on various external factors renders STLF a challenging task. To that end, several deep learning models have been proposed in the literature for STLF, reporting promising results. In order to evaluate the accuracy of said models in day-ahead forecasting settings, in this paper we focus on the national net aggregated STLF of Portugal and conduct a comparative study considering a set of indicative, well-established deep autoregressive models, namely multi-layer perceptrons (MLP), long short-term memory networks (LSTM), neural basis expansion coefficient analysis (N-BEATS), temporal convolutional networks (TCN), and temporal fusion transformers (TFT). Moreover, we identify factors that significantly affect the demand and investigate their impact on the accuracy of each model. Our results suggest that N-BEATS consistently outperforms the rest of the examined models. MLP follows, providing further evidence towards the use of feed-forward networks over relatively more sophisticated architectures. Finally, certain calendar and weather features like the hour of the day and the temperature are identified as key accuracy drivers, providing insights regarding the forecasting approach that should be used per case.",
    "id": "2302.12168v2"
  }
]

print(f"Loaded {len(ABSTRACTS)} abstracts:")
for i, a in enumerate(ABSTRACTS, 1):
    print(f"  {i:2d}. [{a['field']}] {a['title'][:70]}")

### 3b — Run the batch (one API call per abstract → structured rows)

This is the loop you'll reuse on your own data. It prints progress as it goes
(12 calls, usually 30–90 s). If a response isn't valid JSON, it retries once.
Some models wrap the JSON object in code fences; the loop removes those before
it parses.

In [ ]:
import json, time

SYSTEM_PROMPT = (
    "You are a research literature assistant. Read the abstract and respond with "
    "ONLY a valid JSON object (no markdown, no commentary) with exactly these keys: "
    '{"summary": "<one-sentence summary>", '
    '"method": "<the main method or approach, one short phrase>", '
    '"result": "<the key result or contribution, one short phrase>"}. '
    'If a detail is not stated in the abstract, set it to "not stated". '
    "Return ONLY the JSON object."
)

def extract(record):
    # One model call for one abstract -> dict (or a PARSE-FAILED marker).
    user = (f"Field: {record['field']}\nTitle: {record['title']}\n"
            f"Abstract: {record['abstract']}")
    for attempt in range(2):   # one retry if the JSON is malformed
        resp = client.chat.completions.create(
            model=MODEL,
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": user},
            ],
            temperature=0,
        )
        text = resp.choices[0].message.content.strip()
        # Strip any ```json ... ``` fence some models add around the object
        t = text
        if t.startswith("```"):
            t = t.removeprefix("```json").removeprefix("```").removesuffix("```").strip()
        try:
            return json.loads(t)
        except json.JSONDecodeError:
            if attempt == 0:
                continue
            return {"summary": "PARSE FAILED", "method": text[:80], "result": ""}

rows, failures = [], 0
for i, rec in enumerate(ABSTRACTS, 1):
    row = extract(rec)
    ok = bool(row) and row.get("summary") != "PARSE FAILED"
    if not ok:
        failures += 1
        rows.append({"field": rec["field"], "title": rec["title"], "arxiv_id": rec["id"],
                     "summary": "PARSE FAILED", "method": "", "result": ""})
        print(f"  {i:2d}/{len(ABSTRACTS)}  {rec['field']:<22} -> PARSE FAILED (kept raw snippet)")
    else:
        rows.append({"field": rec["field"], "title": rec["title"], "arxiv_id": rec["id"],
                     **{k: row.get(k, "not stated") for k in ("summary", "method", "result")}})
        print(f"  {i:2d}/{len(ABSTRACTS)}  {rec['field']:<22} -> ok")

print(f"\nDone. {len(rows)} rows, {failures} parse failure(s).")

### 3c — Save & inspect the CSV

The rows are written to `triage_table.csv` — one row per paper, ready for a
spreadsheet. `csv.DictWriter` handles quoting, so commas inside a summary won't
shift the columns.

In [ ]:
import csv

out = "triage_table.csv"
fields = ["field", "title", "arxiv_id", "summary", "method", "result"]
with open(out, "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=fields)
    w.writeheader()
    w.writerows(rows)
print(f"Wrote {len(rows)} rows to {out}\n")

# Show a readable slice
for r in rows[:4]:
    print(f"[{r['field']}] {r['title'][:55]}")
    print(f"   summary: {r['summary'][:100]}")
    print(f"   method:  {r['method'][:80]}")
    print()

### The loop generalizes

The same batch call works on grant proposals, lab notes, instrument logs —
anything you can turn into a list of texts.


### Key Points

- **Structured output** = a `system` prompt that forces a strict JSON *shape* (keys you define).
- Be defensive: strip code fences, retry once on a bad parse, keep a `PARSE FAILED` marker.
- `csv.DictWriter` writes clean, quote-safe CSV.

## Experiments & Tools Around the API

### Questions

- What actually changes the model's answer — the prompt, or the model?
- How do I try my own data with the same batch loop?
- What other tools can I point at the same key and endpoint?


### Objectives

- Systematically vary one variable at a time (prompt or model).
- Extend the JSON schema with new fields.
- Batch-process your own (non-sensitive) data.
- Name the researcher-facing tools that reuse the same OpenAI-compatible API.


## Guided experiment time

Pick whichever cells fit your own research. Each one changes **exactly one
variable** — that's how you learn what actually matters.


**Before Experiment 4:** do not send sensitive/regulated data — no health
records, no export-controlled work, no SSNs, no personal identifiers. If unsure,
ask your institution first.

### Experiment 1 — Vary the prompt (same abstract, 3 asks)

In [ ]:
rec = ABSTRACTS[0]   # pick any index 0..11

asks = [
    "Summarize this abstract in one sentence.",
    "Summarize this abstract in one sentence, then list up to 3 limitations the authors mention or that are implied.",
    "Explain the core idea of this abstract to a first-year PhD student in a different field. Keep it under 80 words.",
]
for i, ask in enumerate(asks, 1):
    resp = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": "You are a helpful research assistant."},
            {"role": "user", "content": f"{ask}\n\nAbstract: {rec['abstract']}"},
        ],
    )
    print(f"--- Ask {i}: {ask[:60]}...")
    print(resp.choices[0].message.content.strip()[:300])
    print()

### Experiment 2 — Vary the model (same request, 2 models)

Set `B` to a second model ID from the list printed in the first-call episode.
Watch style, length, and what each gets right or wrong.

In [ ]:
A = MODEL
B = None   # <-- set me to another model ID, e.g. B = 'gpt-4o' (must be in your list)

if not B:
    raise SystemExit("Set B = 'some-model-id' (from the list in Section 2) and re-run.")

prompt = ("In under 60 words, what is the main method and the key result of this "
          f"abstract?\n\nAbstract: {ABSTRACTS[1]['abstract']}")

for label, m in (("Model A", A), ("Model B", B)):
    try:
        resp = client.chat.completions.create(
            model=m,
            messages=[{"role": "user", "content": prompt}],
            temperature=0,
        )
        print(f"=== {label} ({m}) ===")
        print(resp.choices[0].message.content.strip())
        print()
    except Exception as e:
        print(f"=== {label} ({m}) === FAILED: {e}\n")

### Experiment 3 — Tighten the structure (add a field to the schema)

Add `reproducible` and `confidence` keys and watch the model fill them:

In [ ]:
SYSTEM2 = (
    "Respond with ONLY a valid JSON object (no other text) with exactly these keys: "
    '{"summary": "<one sentence>", '
    '"method": "<main method, short>", '
    '"result": "<key result, short>", '
    '"reproducible": "yes | no | maybe", '
    '"confidence": "high | medium | low"}. '
    "Use 'not stated' where the abstract gives no information. Return ONLY the JSON."
)

for rec in ABSTRACTS[:3]:
    resp = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": SYSTEM2},
            {"role": "user", "content": f"Title: {rec['title']}\nAbstract: {rec['abstract']}"},
        ],
        temperature=0,
    )
    print(rec["title"][:60])
    print(resp.choices[0].message.content.strip())
    print()

### Experiment 4 — Your own data

Paste 3–5 short paragraphs from your own work into `MY_TEXTS` and run the same batch loop. *(Only send data that is fine to send.)*

In [ ]:
MY_TEXTS = [
    "Paste paragraph 1 here",
    "Paste paragraph 2 here",
    "Paste paragraph 3 here",
]

if all(t.startswith("Paste") for t in MY_TEXTS):
    raise SystemExit("Replace the MY_TEXTS list with your own text, then re-run.")

for i, text in enumerate(MY_TEXTS, 1):
    resp = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": f"Text: {text}"},
        ],
        temperature=0,
    )
    print(f"--- Text {i} ---")
    print(resp.choices[0].message.content.strip())
    print()

## Tools around the API

Because the API is **OpenAI-compatible**, a set of ready-made tools works with
the **same key and the same endpoint**. A quick tour:

- **OpenCode** — a terminal/desktop coding assistant. ASU RC has a setup guide,
  and Voyager even generates the provider config for you.
- **VS Code** — *Chat → Manage Language Models → Add Models → Custom Endpoint*;
  paste [`https://openai.rc.asu.edu/v1`](https://openai.rc.asu.edu/v1) + your key.
- **Jupyter AI** — the `%ai` magic, right inside your notebooks, same gateway.
- **AnvilGPT (Purdue)** — [anvilgpt.rcac.purdue.edu](https://anvilgpt.rcac.purdue.edu), a separate
  service with its own access request, but the mental model is identical (it's
  RAG over a vector DB).

**Pick the shape that fits the task:** raw script → the API directly; coding →
OpenCode/VS Code; notebook work → Jupyter AI.

## Where to go next (same API, deeper)

- **Streaming** — `client.chat.completions.create(..., stream=True)`; watch tokens arrive.
- **Function / tool calling** — let the model call *your* Python functions.
- **RAG** — ground answers in your documents; that is the next episode
  (and what AnvilGPT does out of the box).
- **Canonical docs:** [docs.rc.asu.edu/ai/api](https://docs.rc.asu.edu/ai/api).

### Data governance — read before you paste

Do **not** send data your institution would call sensitive or proprietary.
Never send regulated data (HIPAA records, export-controlled work, SSNs,
biometrics, personal identifiers). The gateway is a convenience, **not** a
secure enclave.

### Key Points

- Change **one variable at a time** — prompt or model — and compare the outputs.
- You can add fields to the JSON schema and the model will fill them.
- The same loop runs on your own (non-sensitive) data.
- OpenAI-compatible means OpenCode, VS Code, Jupyter AI, and AnvilGPT all reuse the same key + endpoint.

## A Simple RAG Example

### Questions

- How do I make a model answer from *my* documents?
- Why does a vector database find the right paper even when no words match?


### Objectives

- Explain retrieval-augmented generation (RAG) in one sentence.
- Store the 12 abstracts in a vector database and search them by meaning.
- Put the retrieved passages into the prompt so the model answers from them.


## What RAG is

So far the model has answered from what it saw in its training data. With
**RAG** (**R**etrieval-**A**ugmented **G**eneration) you hand it the source
text *with the question* and ask it to answer from that text. The idea has
two steps:

1. **Retrieve** — find the documents (or passages) most relevant to the
   question.
2. **Generate** — ask the model to answer using only the passages you found.

Production systems (like Purdue's AnvilGPT) do the retrieving with a **vector
database**: a store that finds text by *meaning*. This episode builds one for
real, over the 12 abstracts from the *Build the research tool* episode, using
[ChromaDB](https://www.trychroma.com) — an open-source vector database that
installs with one `pip`.

You need `ABSTRACTS`, `client`, and `MODEL` from the earlier episodes, so run
*Build the research tool* (or the notebook's section 3a) first.

## Step 1 — Store: put the abstracts in a vector database

First install the library. On Anvil, re-run this cell after every server
restart — installed packages do not survive the session. The first install
takes a minute or two; it is worth the wait.

In [ ]:
%pip install chromadb

An **embedding** is a list of numbers that represents what a piece of text
*says*. Texts about similar things get similar numbers, so a paper about
"urban air pollution" lands near a question about "city air quality" even
though the words share nothing. A **vector database** stores one embedding
per document and returns the closest ones to whatever you query with.

In [ ]:
import chromadb
from chromadb.config import Settings

chroma = chromadb.EphemeralClient(settings=Settings(anonymized_telemetry=False))
collection = chroma.get_or_create_collection("abstracts")

if collection.count() == 0:   # add only once; re-running the cell is safe
    collection.add(
        documents=[a["abstract"] for a in ABSTRACTS],
        metadatas=[{"title": a["title"], "field": a["field"]} for a in ABSTRACTS],
        ids=[a["id"] for a in ABSTRACTS],
    )
print(collection.count(), "abstracts stored")

Three things to notice. `EphemeralClient` keeps everything in memory — it is
gone when the kernel stops, which is exactly right for a lesson. `add()`
embeds each abstract (the first run downloads a small embedding model, about
80 MB, so give it a minute) and stores it with the metadata you attach. And
`ids` must be unique — they are how the database tells documents apart.

## Step 2 — Retrieve: ask by meaning

In [ ]:
QUESTION = "Which of these papers uses satellite data, and what is it trying to predict?"
result = collection.query(query_texts=[QUESTION], n_results=3)

for meta, distance in zip(result["metadatas"][0], result["distances"][0]):
    print(f"{distance:.2f}  [{meta['field']}] {meta['title'][:70]}")

```output
1.11  [Environmental Science] Urban Air Pollution Forecasting: a Machine Learning Approach leveraging Satelli
1.42  [Environmental Science] Predicting concentration levels of air pollutants by transfer learning and recur
1.51  [Energy] Lasso estimation for GEFCom2014 probabilistic electric load forecasting
```

`distance` is a dissimilarity score: **lower means closer** to the question.
The `title` and `field` come from the metadata you attached in `add()`. Your
distance numbers may differ slightly from the ones shown — they depend on the
embedding model version — but the ranking holds.

## Step 3 — Generate: answer from the passages you found

Now build a prompt that contains (a) an instruction to answer *only* from the
passages, and (b) the passages themselves. Nothing else changes — same
`client`, same `MODEL`, same `create()` call:

In [ ]:
def rag_answer(question, k=3):
    result = collection.query(query_texts=[question], n_results=k)
    passages = "\n\n".join(
        f"[{i+1}] ({meta['field']}) {meta['title']}\n{doc}"
        for i, (meta, doc) in enumerate(zip(result["metadatas"][0],
                                            result["documents"][0]))
    )
    prompt = (
        "Answer the question using ONLY the passages below. Cite the passage "
        "number(s) [1], [2], ... for each claim. If the passages do not contain "
        "the answer, say so.\n\n"
        f"Question: {question}\n\nPassages:\n{passages}"
    )
    resp = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
    )
    return resp.choices[0].message.content

print(rag_answer(QUESTION))

You should get an answer that names the satellite paper and says what it
predicts, with citations like `[1]`.

### What you just built

That is a real, if tiny, RAG pipeline. The pieces map directly onto the big
systems:

| This episode | A production RAG system |
|---|---|
| a ChromaDB collection | a vector database (Chroma, pgvector, FAISS, ...) |
| `collection.add(...)` — embed and store | an ingestion pipeline |
| `collection.query(...)` — nearest passages | the retriever |
| `rag_answer()` — passages + question in the prompt | the same: context in the prompt |

The `generate` step is identical in both. Most of the engineering in RAG
lives in the `retrieve` step — finding the right passages before the model
ever sees the question.

### Note that

- **The default embedding model is small and general.** It works well here;
  for specialized text (medical jargon, law), pick an embedding model trained
  on that domain.
- **We embed whole abstracts.** Long documents must be split into passages
  (**chunking**) first, and chunk size is a real decision. Our 12 abstracts
  are already short enough.

A plain word-overlap search also works at this size — try it as an exercise
below and compare. The vector database earns its keep when synonyms and
paraphrase start mattering.

### Exercises

**Find a match with no shared words.** Query the collection with questions
that use none of the words in any abstract, for example:

In [ ]:
for q in ["What do these papers say about air quality in cities?",
          "How can hospitals make MRI scans from different machines comparable?",
          "Which paper predicts electricity demand?"]:
    result = collection.query(query_texts=[q], n_results=1)
    m = result["metadatas"][0][0]
    print(f"{result['distances'][0][0]:.2f}  [{m['field']}] {m['title'][:70]}")

The first finds the pollution papers, the second the MRI-harmonization
papers, the third the load-forecasting papers — none of the question words
appear in those titles. That is meaning-based search. **Stretch:** create a
second collection with 5–10 pieces of your own writing (meeting notes, your
lab's SOPs — anything non-sensitive) and query it.

### Key Points

- RAG = **retrieve** the relevant passages, then **generate** an answer from
  them.
- A vector database finds text by meaning: embeddings turn text into numbers,
  and similar meaning lands nearby.
- The generate step is the same `create()` call as anywhere else in the
  lesson — the passages just join the prompt.